# Figure 6: MLP Hyperparameter Tuning Heatmaps + Training Loss

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import os

plt.rcParams['font.family'] = 'Times New Roman'

# Load Grid Search Results

In [ ]:
# Grid search settings
hidden_sizes = [10, 20, 40, 80]
hidden_layers = [4, 8, 12]
batch_sizes = [200, 2000]
learn_rates = [0.001, 0.01]

output_dir = './data/output_data_e6_bias'

def load_grid_results(target):
    """Load cross-validated NMSE for each (hidden_size, hidden_layers) pair.
    For each pair, report the minimum NMSE across batch sizes and learning rates."""
    results = np.full((len(hidden_sizes), len(hidden_layers)), np.nan)
    
    for i, hs in enumerate(hidden_sizes):
        for j, hl in enumerate(hidden_layers):
            best_nmse = float('inf')
            for bs in batch_sizes:
                for lr in learn_rates:
                    # Filename pattern: output_e6__{target}_{hl}_{hs}_{fold}_{bs}_{lr}
                    fname = f'output_e6__{target}_{hl}_{hs}_1_{bs}_{lr}'
                    fpath = os.path.join(output_dir, fname)
                    if os.path.exists(fpath):
                        with open(fpath, 'rb') as f:
                            data = pickle.load(f)
                        # data shape: (n_checkpoints, 2, 6), index [4] is NMSE
                        # use last checkpoint, test metrics (index 0)
                        nmse = data[-1, 0, 4]
                        if nmse < best_nmse:
                            best_nmse = nmse
            if best_nmse < float('inf'):
                results[i, j] = best_nmse
    
    return results

results_fval = load_grid_results('FValue')
results_w1 = load_grid_results('W1')

# Plot Heatmaps

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(18, 5), gridspec_kw={'width_ratios': [1, 1, 1]})

# phi_tau heatmap
ax0 = axs[0]
sns.heatmap(results_fval * 1e3, annot=True, fmt='.2f', cmap='YlGnBu',
            xticklabels=hidden_layers, yticklabels=hidden_sizes,
            ax=ax0, cbar=True)
ax0.set_xlabel('Layer Depth', fontsize=13)
ax0.set_ylabel('Layer Size', fontsize=13)
ax0.set_title(r'$\phi_{\tau}$ ($\times 10^{-3}$)', fontsize=14)

# Find and highlight best cell
best_idx = np.unravel_index(np.nanargmin(results_fval), results_fval.shape)
ax0.add_patch(plt.Rectangle((best_idx[1], best_idx[0]), 1, 1,
              fill=False, edgecolor='red', linewidth=3))

# Delta_tau heatmap
ax1 = axs[1]
sns.heatmap(results_w1 * 1e3, annot=True, fmt='.1f', cmap='YlGnBu',
            xticklabels=hidden_layers, yticklabels=hidden_sizes,
            ax=ax1, cbar=True)
ax1.set_xlabel('Layer Depth', fontsize=13)
ax1.set_ylabel('')
ax1.set_title(r'$\Delta_{\tau}$ ($\times 10^{-3}$)', fontsize=14)

best_idx_w1 = np.unravel_index(np.nanargmin(results_w1), results_w1.shape)
ax1.add_patch(plt.Rectangle((best_idx_w1[1], best_idx_w1[0]), 1, 1,
              fill=False, edgecolor='red', linewidth=3))

# Training loss curve for selected Delta_tau model
ax2 = axs[2]
# Load training loss for best Delta_tau model
best_hl_w1 = hidden_layers[best_idx_w1[1]]
best_hs_w1 = hidden_sizes[best_idx_w1[0]]
loss_file = os.path.join(output_dir, f'output_e6__W1_{best_hl_w1}_{best_hs_w1}_1_200_0.01')

if os.path.exists(loss_file):
    with open(loss_file, 'rb') as f:
        loss_data = pickle.load(f)
    # Plot training NMSE over epochs (index 4 = NMSE, index 0 = test metrics)
    epochs = np.arange(1, loss_data.shape[0]+1) * 50  # epoch_chunk_size=50
    ax2.semilogy(epochs, loss_data[:, 0, 4])
    ax2.set_xlabel('Epoch', fontsize=13)
    ax2.set_ylabel('NMSE', fontsize=13)
    ax2.set_title('Training Loss', fontsize=14)
else:
    ax2.text(0.5, 0.5, 'Loss data not found', transform=ax2.transAxes, ha='center')

fig.suptitle('MLP neural network hyperparameter tuning', fontsize=18, fontweight='bold')
plt.tight_layout()
plt.savefig('Figure6.png', dpi=300, bbox_inches='tight')
plt.show()